# Parse Video Data

In [ ]:
!pip install ffmpeg-python pillow
!git clone https://github.com/soCzech/TransNetV2.git
%cd TransNetV2/inference

In [ ]:
# Import module
import os
import cv2
import json
import glob
import ffmpeg
import torch
import numpy as np
from tqdm import tqdm
from transnetv2 import TransNetV2

In [5]:
videos_dir = '/kaggle/input/news-event-retrieval-video-data'
all_video_paths = dict()
for folder_name in sorted(os.listdir(videos_dir)):
    # if part != "Videos_L21_a": continue
    data_part = folder_name.replace('Videos_', '') # L21_a for ex
    all_video_paths[data_part] =  dict()

for data_part in sorted(all_video_paths.keys()):
    data_part_path = f'{videos_dir}/Videos_{data_part}/video'
    video_paths = sorted(os.listdir(data_part_path))
    video_ids = [video_path.replace('.mp4', '').split('_')[-1] for video_path in video_paths]
    for video_id, video_path in zip(video_ids, video_paths):
        video_path_full = f'{data_part_path}/{video_path}'
        all_video_paths[data_part][video_id] = video_path_full

In [7]:
all_video_paths['L21_a']

{'V001': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V001.mp4',
 'V002': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V002.mp4',
 'V003': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V003.mp4',
 'V005': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V005.mp4',
 'V006': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V006.mp4',
 'V007': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V007.mp4',
 'V008': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V008.mp4',
 'V009': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V009.mp4',
 'V010': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V010.mp4',
 'V011': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V011.mp4',
 'V012': '/kaggle/input/news-event-retrieval-video-data/Videos_L21_a/video/L21_V012.mp4',
 'V013': '

Upload manually model and variables files

In [7]:
# !cp /kaggle/input/transnetv2/tensorflow2/default/1/saved_model.pb /kaggle/working/TransNetV2/inference/transnetv2-weights/

In [9]:
# ! cp /kaggle/input/variables/variables.data-00000-of-00001 /kaggle/working/TransNetV2/inference/transnetv2-weights/variables
# ! cp /kaggle/input/variables/variables.index /kaggle/working/TransNetV2/inference/transnetv2-weights/variables

# Inference

In [ ]:
model = TransNetV2()

save_dir = '/kaggle/working/SceneJSON'
if not os.path.exists(save_dir):
    os.mkdir(save_dir)

for key, video_paths_dict in tqdm(all_video_paths.items()):
    video_ids = sorted(video_paths_dict.keys())
    
    if not os.path.exists(os.path.join(save_dir, key)):
        os.mkdir(os.path.join(save_dir, key))
    
    for video_id in tqdm(video_ids):
        video_path = video_paths_dict[video_id]
        _, single_frame_predictions, _ = model.predict_video(video_path)
        
        # Generate list of scenes from predictions, returns tuples of (start frame, end frame)
        scenes = model.predictions_to_scenes(single_frame_predictions)
        
        with open(f"{save_dir}/{key}/{video_id}.json", 'w') as f:
            json.dump(scenes.tolist(), f)